In [ ]:
import json
import pdfplumber

In [ ]:
def parse_json_response(raw):
    clean = raw.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    return json.loads(clean)

In [ ]:
def extract_text_from_pdf(filepath):
    """Extract text from a PDF file using pdfplumber."""
    text_chunks = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_chunks.append(page_text)
    return "\n".join(text_chunks)

In [ ]:
def print_checklist(crime_type):
    from data_loader import load_checklist

    checklist = load_checklist(crime_type)
    print(f"\n{checklist['crime_type']} — {len(checklist['required_elements'])} elements\n")
    for element in checklist["required_elements"]:
        print(f"  {element['id']} — {element['element']}")

In [ ]:
def print_scenario(scenario, checklist):
    """Print a scenario in a readable format for debugging."""
    print(f"CRIME TYPE: {scenario['crime_type']}")
    print(f"INTERVIEWEE: {scenario['interviewee_name']} ({scenario['interviewee_role']})")
    print(f"DISPATCH: {scenario['dispatch_line']}")
    print("\nCHECKLIST Q&A:")
    print("-" * 60)
    for el in checklist["required_elements"]:
        question = el["element"]
        answer = scenario["facts"].get(el["id"], "MISSING")
        print(f"\nQ ({el['id']}): {question}")
        print(f"A: {answer}")

In [ ]:
def extract_checklist_from_text(raw_text, crime_type, client, model, temp):
    """Extract structured checklist elements from raw PDF text using LLM."""
    slug = crime_type.lower().replace(" ", "_").replace("/", "_")

    prompt = f"""You are extracting required reporting elements from a police report checklist document.

Crime type: {crime_type}

Source document text:
{raw_text}

Extract every required element an officer must document for this crime type.
Return ONLY a valid JSON object with this exact structure, no markdown fences, no extra text:

{{
  "crime_type": "{crime_type}",
  "required_elements": [
    {{"id": "{slug}_01", "element": "description of required element"}},
    {{"id": "{slug}_02", "element": "description of required element"}}
  ]
}}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        response_format={"type": "json_object"}
    )

    return parse_json_response(response.choices[0].message.content)

In [ ]:
def chunk_narrative(narrative):
    import re

    # Split on any sequence of newlines
    paragraphs = [p.strip() for p in re.split(r'\n+', narrative) if p.strip()]

    chunks = []
    for paragraph in paragraphs:
        if len(paragraph) <= 1000:
            chunks.append(paragraph)
        else:
            # Split long paragraphs at sentence boundaries under 1000 chars
            remaining = paragraph
            while len(remaining) > 1000:
                # Find the last sentence ending before 1000 chars
                boundary = remaining[:1000].rfind('.')
                if boundary == -1:
                    # No sentence boundary found, hard cut at 1000
                    boundary = 1000
                else:
                    boundary += 1  # include the period
                chunks.append(remaining[:boundary].strip())
                remaining = remaining[boundary:].strip()
            if remaining:
                chunks.append(remaining)

    return {i: chunk for i, chunk in enumerate(chunks)}